# Import

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import entropy
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import classification_report
import lightgbm as lgb
import matplotlib.pyplot as plt

import sys
from pathlib import Path

# Add src/ to path (once, so imports work)
sys.path.append(str(Path().resolve().parent / "src"))
# 
# Enable autoreload for Jupyter notebooks
%load_ext autoreload
%autoreload 2

from paths import DATA_DATASETS
from helper_functions import get_master_dataframe
import feature_construction as fc

Failed to read module file 'C:\Users\JuliusAdmin\AppData\Local\Programs\Python\Python312\Lib\urllib\parse.py' for module 'urllib.parse': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\JuliusAdmin\Documents\GitHub\Marketing-Analytics\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\JuliusAdmin\Documents\GitHub\Marketing-Analytics\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\JuliusAdmin\AppData\Local\Programs\Python\Python312\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>",

In [2]:
# Get master dataframe - constructed of transactions, outfits, and outfit_clusters
master = get_master_dataframe()


test_customers = pd.read_csv(DATA_DATASETS / "test_customers.csv", sep=";")
baseline_pred = pd.read_csv(DATA_DATASETS / "pred.csv")

Master shape: (60897, 11)
   customer.id                                outfit.id rentalPeriod.start  \
0         3448  outfit.5c081909537b42239e465d2d615c705f         2023-03-26   
1         2924  outfit.c34969dd8b334064aa90bfb60c8ec308         2023-03-27   

  rentalPeriod.end  cluster         cluster_name  pricePerWeek  pricePerMonth  \
0       2023-04-25      3.0  Sweaters & Knitwear         750.0         1500.0   
1       2023-04-26      7.0      Jackets & Coats         990.0         1980.0   

   retailPrice  duration_days  revenue  
0       2500.0             30   1500.0  
1       3900.0             30   1980.0  


In [3]:
# Fold structure
# TIMEFRAME_START  = pd.Timestamp("2017-04-01")
TRAIN_CUTOFF     = pd.Timestamp("2021-09-06")  # features end here for training
LABEL_END        = pd.Timestamp("2022-09-06")  # labels end here for training
FINAL_CUTOFF     = pd.Timestamp("2023-09-06")  # features end here for testing

X_train, X_test, y_train, y_test, id_train, id_test = fc.generate_train_test_splits(df=master, train_cutoff=TRAIN_CUTOFF, label_end=LABEL_END, final_cutoff=FINAL_CUTOFF)

In [4]:
# Baseline Model Feature Selection: Sequential Forward Selection (SFS) with LightGBM
sfs_selected_features = ['recency', 'monetary', 'avg_revenue', 'weighted_rev', 'monetary_x_frequency', 'tenure_days', 'revenue_per_day', 'rentals_per_day', 'n_summer', 'active_months', 'revenue_trend', 'pct_weekly_rentals', 'cluster_affinity_0', 'cluster_affinity_1', 'cluster_affinity_2', 'cluster_affinity_7', 'max_retail_price', 'avg_price_per_week']

X_train_sfs = X_train[sfs_selected_features]
X_test_sfs = X_test[sfs_selected_features]

# 2-stage model
# Churn Model Feature Selection: Sequential Forward Selection (SFS) with LightGBM for the Classification Model
sfs_selected_features_2s_clf = [
    'recency',
    'monetary_x_frequency',
    'pct_weekly_rentals',
    'cluster_affinity_6'
]

X_train_sfs_clf = X_train[sfs_selected_features_2s_clf]
X_test_sfs_clf = X_test[sfs_selected_features_2s_clf]

# Regression Model Feature Selection: Sequential Forward Selection (SFS) with LightGBM for the Regression Model
# sfs_selected_features_2s_reg = ['recency', 'monetary', 'tenure_days', 'avg_days_between_rentals', 'revenue_trend']
sfs_selected_features_2s_reg = [
    'recency', 
    'monetary', 
    'avg_revenue', 
    'weighted_rev', 
    'monetary_x_frequency', 
    'tenure_days', 
    'revenue_per_day', 
    'rentals_per_day', 
    'revenue_trend', 
    'max_retail_price', 
    'avg_price_per_week'
]

X_train_sfs_reg = X_train[sfs_selected_features_2s_reg]
X_test_sfs_reg = X_test[sfs_selected_features_2s_reg]

In [5]:
# Fit and evaluate
lgbm_model = lgb.LGBMRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_samples = 20,
    subsample         = 0.8,
    random_state      = 42,
    verbose           = -1,

    objective="tweedie",
    tweedie_variance_power=1.3
)

# Train fold
lgbm_model.fit(X_train_sfs, y_train)

train_predictions = np.clip(np.array(lgbm_model.predict(X_train_sfs)), 0, None)
train_mae = mean_absolute_error(y_train, train_predictions)
print(f"MAE on training fold: {train_mae:.2f} NOK")


# Test fold
test_predictions = np.clip(np.array(lgbm_model.predict(X_test_sfs)), 0, None)

test_mae = mean_absolute_error(y_test, test_predictions)
print(f"MAE on test fold: {test_mae:.2f} NOK")

MAE on training fold: 377.46 NOK
MAE on test fold: 2499.41 NOK


## Two-stage Model

In [6]:
# Stage 1: Who is going to be active?
y_train_churn = (y_train > 0).astype(int).values.ravel()
y_test_churn  = (y_test  > 0).astype(int).values.ravel()

n_inactive = (y_train_churn == 0).sum()
n_active   = (y_train_churn == 1).sum()
print(f"Training: {n_active} aktiv, {n_inactive} inaktiv ({n_inactive/n_active:.1f}:1)")

churn_model = lgb.LGBMClassifier(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_samples = 20,
    subsample         = 0.8,
    # scale_pos_weight  = n_inactive / n_active,
    random_state      = 42,
    verbose           = -1
)
# churn_model.fit(X_train_sfs_clf, y_train_churn)

# churn_prob_test  = churn_model.predict_proba(X_test_sfs_clf)[:, 1]
# churn_pred_test  = churn_model.predict(X_test_sfs_clf)

churn_model.fit(X_train, y_train_churn)

churn_prob_test  = churn_model.predict_proba(X_test)[:, 1]
churn_pred_test  = churn_model.predict(X_test)


print("Churn Classifier — Test Fold:")
print(classification_report(y_test_churn, churn_pred_test, target_names=["Inactive", "Active"]))

print("Churn Classifier — Test Fold - with optimised threshold:")
churn_pred_test_opt = (churn_prob_test >= 0.85).astype(int)
print(classification_report(y_test_churn, churn_pred_test_opt, target_names=["Inactive", "Active"]))

# Stage 2: How much revenue do active customers generate?
active_mask = (y_train > 0).values.ravel()

print(f"\nRevenue Model trained on {active_mask.sum()} active customers.")

# Log-transform the target for better modeling (optional, but often helps with skewed revenue data)
# y_train_log = np.log1p(y_train.values.ravel()[active_mask])

revenue_model = lgb.LGBMRegressor(
    n_estimators      = 600,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_samples = 0,
    subsample         = 0.8,
    random_state      = 42,
    verbose           = -1,

    objective="tweedie",
    tweedie_variance_power=1.8
)

# revenue_model.fit(X_train[active_mask], y_train_log)
revenue_model.fit(X_train_sfs_reg[active_mask], y_train.values.ravel()[active_mask])
# revenue_model.fit(X_train_sfs_reg[active_mask], y_train_log)

# Predict and transform back from log scale
pred_log = np.array(revenue_model.predict(X_test_sfs_reg))
revenue_if_active = np.clip(pred_log, 0, None)
#revenue_if_active = np.clip(np.expm1(pred_log), 0, None)

# Hard: entweder 0 oder predicted revenue
final_pred_hard = np.where(churn_pred_test == 1, revenue_if_active, 0)
final_pred_hard_opt = np.where(churn_pred_test_opt == 1, revenue_if_active, 0)

# Soft: P(aktiv) × expected revenue — oft besser für MAE
final_pred_soft = churn_prob_test * revenue_if_active

mae_hard      = mean_absolute_error(y_test, final_pred_hard)
mae_hard_opt  = mean_absolute_error(y_test, final_pred_hard_opt)
mae_soft      = mean_absolute_error(y_test, final_pred_soft)
mae_baseline  = mean_absolute_error(y_test, np.clip(np.array(lgbm_model.predict(X_test_sfs)), 0, None))

print(f"\nResults on Test Fold:")
print(f"Single-Stage LightGBM: {mae_baseline:.2f} NOK")
print(f"Two-Stage Hard: {mae_hard:.2f} NOK")
print(f"Two-Stage Hard (opt. threshold): {mae_hard_opt:.2f} NOK")
print(f"Two-Stage Soft: {mae_soft:.2f} NOK")

Training: 439 aktiv, 5504 inaktiv (12.5:1)
Churn Classifier — Test Fold:
              precision    recall  f1-score   support

    Inactive       0.98      0.98      0.98      6193
      Active       0.78      0.79      0.78       616

    accuracy                           0.96      6809
   macro avg       0.88      0.88      0.88      6809
weighted avg       0.96      0.96      0.96      6809

Churn Classifier — Test Fold - with optimised threshold:
              precision    recall  f1-score   support

    Inactive       0.97      0.99      0.98      6193
      Active       0.86      0.72      0.78       616

    accuracy                           0.96      6809
   macro avg       0.92      0.86      0.88      6809
weighted avg       0.96      0.96      0.96      6809


Revenue Model trained on 439 active customers.

Results on Test Fold:
Single-Stage LightGBM: 2499.41 NOK
Two-Stage Hard: 2543.56 NOK
Two-Stage Hard (opt. threshold): 2438.79 NOK
Two-Stage Soft: 2656.61 NOK


In [7]:
y_test_flat = y_test.values.ravel()

# Diagnostics
results = pd.DataFrame({
    "actual":         y_test_flat,
    "pred_single":    np.clip(np.array(lgbm_model.predict(X_test_sfs)), 0, None),
    "pred_hard":      final_pred_hard,
    "pred_hard_opt":  final_pred_hard_opt,
    "pred_soft":      final_pred_soft,
    "churn_prob":     churn_prob_test,
    "error_single":   np.abs(y_test_flat - np.clip(np.array(lgbm_model.predict(X_test_sfs)), 0, None)),
    "error_hard":     np.abs(y_test_flat - final_pred_hard),
    "error_hard_opt": np.abs(y_test_flat - final_pred_hard_opt),
    "error_soft":     np.abs(y_test_flat - final_pred_soft),
})

results["bucket"] = pd.cut(results["actual"],
    bins=[-0.01, 0.01, 1000, 5000, 15000, 30000, 45000, 60000, 999999],
    labels=["Churned (0)", "1–1k NOK", "1k–5k NOK", "5k-15k NOK", "15k-30k NOK", "30k-45k NOK", "45k-60k NOK", "60k+ NOK"])

print("\nMAE for each customer segment:")
print(results.groupby("bucket", observed=True).agg(
    n           = ("actual",      "count"),
    mae_single  = ("error_single","mean"),
    mae_hard    = ("error_hard",  "mean"),
    mae_hard_opt = ("error_hard_opt", "mean"),
    mae_soft    = ("error_soft",  "mean"),
    avg_actual  = ("actual",      "mean"),
).round(2).to_string())

# Threshold Optimisation
# Find optimal threshold for churn classifier to minimize MAE
print("\nThreshold Optimization:")
thresholds = np.arange(0.5, 1, 0.01)
threshold_results = []

for t in thresholds:
    pred = np.where(churn_prob_test >= t, revenue_if_active, 0)
    mae  = mean_absolute_error(y_test, pred)
    threshold_results.append({"threshold": t, "mae": mae})

thresh_df = pd.DataFrame(threshold_results)
best_thresh = thresh_df.loc[thresh_df["mae"].idxmin(), "threshold"]
best_mae    = thresh_df["mae"].min()

print(thresh_df.to_string(index=False))
print(f"\nBest Threshold: {best_thresh:.2f} → MAE: {best_mae:.2f} NOK")


MAE for each customer segment:
                n  mae_single  mae_hard  mae_hard_opt  mae_soft  avg_actual
bucket                                                                     
Churned (0)  6193      409.15    393.91        250.92    540.66        0.00
1k–5k NOK      84     9271.41  13403.54      11374.80  12139.74     2951.35
5k-15k NOK    122    13290.47  13783.67      13696.00  13064.33     9855.13
15k-30k NOK   103    15932.48  17056.72      18061.88  16906.12    21818.84
30k-45k NOK   102    21528.81  18462.20      18804.36  18360.14    38050.78
45k-60k NOK    96    29218.38  26781.85      28732.88  27502.66    51650.07
60k+ NOK      109    49930.59  53771.01      54023.59  53877.39    96566.65

Threshold Optimization:
 threshold         mae
      0.50 2543.560023
      0.51 2537.291104
      0.52 2535.604636
      0.53 2535.604636
      0.54 2535.421628
      0.55 2536.456230
      0.56 2528.669974
      0.57 2513.178244
      0.58 2508.244468
      0.59 2512.199731
      

**Hybrid Approach**

Use churn classifier but then take the Single-Stage Modell predictions and apply churn mask on it.

In [8]:
# Take predictions from the single-stage model
pred_single = np.clip(np.array(lgbm_model.predict(X_test_sfs)), 0, None)

# HYBRID: If Stage 1 says "Active", take the baseline prediction. Otherwise, use 0.
final_pred_hybrid = np.where(churn_pred_test_opt == 1, pred_single, 0)

# Calculate MAE for the hybrid model
mae_hybrid = mean_absolute_error(y_test, final_pred_hybrid)
print(f"Hybrid Two-Stage MAE: {mae_hybrid:.2f} NOK")

# Segmented evaluation for the hybrid model
results_hybrid = pd.DataFrame({
    "actual":         y_test.values.ravel(),
    "pred_hybrid":    final_pred_hybrid,
    "error_hybrid":   np.abs(y_test.values.ravel() - final_pred_hybrid)
})
results_hybrid["bucket"] = pd.cut(results_hybrid["actual"],
    bins=[-0.01, 0.01, 1000, 5000, 15000, 30000, 45000, 60000, 999999],
    labels=["Churned (0)", "1–1k NOK", "1k–5k NOK", "5k-15k NOK", "15k-30k NOK", "30k-45k NOK", "45k-60k NOK", "60k+ NOK"])

print("\nMAE for each customer segment (HYBRID):")
print(results_hybrid.groupby("bucket", observed=True).agg(
    n             = ("actual", "count"),
    mae_hybrid    = ("error_hybrid", "mean"),
    avg_actual    = ("actual", "mean"),
).round(2).to_string())

Hybrid Two-Stage MAE: 2387.40 NOK

MAE for each customer segment (HYBRID):
                n  mae_hybrid  avg_actual
bucket                                   
Churned (0)  6193      237.63        0.00
1k–5k NOK      84     9068.22     2951.35
5k-15k NOK    122    13934.32     9855.13
15k-30k NOK   103    16747.20    21818.84
30k-45k NOK   102    21908.00    38050.78
45k-60k NOK    96    30143.29    51650.07
60k+ NOK      109    50175.18    96566.65


In [9]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd

# Mask for active customers (those with revenue > 0) in the training set
active_mask = (y_train > 0).values.ravel()

# Log transform target variable for better modeling (especially if revenue is skewed)
y_train_log = np.log1p(y_train.values.ravel()[active_mask])

# Create pipeline with RidgeCV to automatically find the best alpha for regularization
ridge_pipeline = make_pipeline(
    SimpleImputer(strategy='median'), 
    StandardScaler(),
    RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0])
)

# Train ridge regression on the active customers only, using the log-transformed revenue as target
ridge_pipeline.fit(X_train_sfs_reg[active_mask], y_train_log)

# Predict and transform back from log scale
pred_log_ridge = ridge_pipeline.predict(X_test_sfs_reg)
revenue_if_active_ridge = np.clip(np.expm1(pred_log_ridge), 0, None)



# Combine Stage 1 and Stage 2 predictions to get final revenue predictions for the test set

# Hard voting: If Stage 1 says "Active" (1), take the Ridge prediction, otherwise 0
final_pred_hard_ridge = np.where(churn_pred_test_opt == 1, revenue_if_active_ridge, 0)
final_pred_soft_ridge = churn_prob_test * revenue_if_active_ridge

# Evaluation
mae_hard_ridge = mean_absolute_error(y_test, final_pred_hard_ridge)
mae_soft_ridge = mean_absolute_error(y_test, final_pred_soft_ridge)

print(f"Results on Test Fold with Ridge Stage 2:")
print(f"Two-Stage Hard (Ridge): {mae_hard_ridge:.2f} NOK")
print(f"Two-Stage Soft (Ridge): {mae_soft_ridge:.2f} NOK")

# Segmented evaluation for the Ridge two-stage model
results_ridge = pd.DataFrame({
    "actual":         y_test.values.ravel(),
    "pred_hard_ridge":  final_pred_hard_ridge,
    "error_hard_ridge": np.abs(y_test.values.ravel() - final_pred_hard_ridge)
})
results_ridge["bucket"] = pd.cut(results_ridge["actual"],
    bins=[-0.01, 0.01, 1000, 5000, 15000, 30000, 45000, 60000, 999999],
    labels=["Churned (0)", "1–1k NOK", "1k–5k NOK", "5k-15k NOK", "15k-30k NOK", "30k-45k NOK", "45k-60k NOK", "60k+ NOK"])

print("\nMAE for each customer segment (Ridge Stage 2):")
print(results_ridge.groupby("bucket", observed=True).agg(
    n             = ("actual", "count"),
    mae_hard_ridge= ("error_hard_ridge", "mean"),
    avg_actual    = ("actual", "mean"),
).round(2).to_string())

Results on Test Fold with Ridge Stage 2:
Two-Stage Hard (Ridge): 2546.38 NOK
Two-Stage Soft (Ridge): 2810.52 NOK

MAE for each customer segment (Ridge Stage 2):
                n  mae_hard_ridge  avg_actual
bucket                                       
Churned (0)  6193          256.64        0.00
1k–5k NOK      84        10667.34     2951.35
5k-15k NOK    122        11391.70     9855.13
15k-30k NOK   103        14271.98    21818.84
30k-45k NOK   102        21566.86    38050.78
45k-60k NOK    96        29892.59    51650.07
60k+ NOK      109        63518.66    96566.65


## Export Churn and Revenue Prediction

In [ ]:
# TIMEFRAME_START  = pd.Timestamp("2016-03-11")
TRAIN_CUTOFF     = pd.Timestamp("2022-09-06")  # features end here for training
LABEL_END        = pd.Timestamp("2023-09-06")  # labels end here for training
FINAL_CUTOFF     = pd.Timestamp("2024-09-06")  # date not in the data, but we set it to one year after the label end to simulate a real test fold

X_train, X_test, y_train, y_test, id_train, id_test = fc.generate_train_test_splits(
    df=master, 
    train_cutoff=TRAIN_CUTOFF, 
    label_end=LABEL_END, 
    final_cutoff=FINAL_CUTOFF)


# Baseline Model Feature Selection: Sequential Forward Selection (SFS) with LightGBM
sfs_selected_features = ['recency', 'monetary', 'avg_revenue', 'weighted_rev', 'monetary_x_frequency', 'tenure_days', 'revenue_per_day', 'rentals_per_day', 'n_summer', 'active_months', 'revenue_trend', 'pct_weekly_rentals', 'cluster_affinity_0', 'cluster_affinity_1', 'cluster_affinity_2', 'cluster_affinity_7', 'max_retail_price', 'avg_price_per_week']

X_train_sfs = X_train[sfs_selected_features]
X_test_sfs = X_test[sfs_selected_features]



# Stage 1: Who is going to be active?
y_train_churn = (y_train > 0).astype(int).values.ravel()
y_test_churn  = (y_test  > 0).astype(int).values.ravel()

n_inactive = (y_train_churn == 0).sum()
n_active   = (y_train_churn == 1).sum()
print(f"Training: {n_active} active, {n_inactive} inactive ({n_inactive/n_active:.1f}:1)")

churn_model = lgb.LGBMClassifier(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_samples = 20,
    subsample         = 0.8,
    # scale_pos_weight  = n_inactive / n_active,
    random_state      = 42,
    verbose           = -1
)

# Fit the churn model on the full feature set
churn_model.fit(X_train, y_train_churn)

# Predict probabilities and classes for the test set
churn_prob_test  = churn_model.predict_proba(X_test)[:, 1]
churn_pred_test_opt = (churn_prob_test >= 0.85).astype(int) # Using the optimized threshold from earlier

# Invert the labels to match assignment requirements (1 = Churned, 0 = Active)
pred_churn = 1 - churn_pred_test_opt


# Create a mapping dataframe for safe merging
predictions_df = id_test.copy()
predictions_df['predicted_churn'] = pred_churn


# Take predictions from the single-stage model
pred_single = np.clip(np.array(lgbm_model.predict(X_test_sfs)), 0, None)

# HYBRID: If Stage 1 says "Active", take the baseline prediction. Otherwise, use 0.
final_pred_hybrid = np.where(churn_pred_test_opt == 1, pred_single, 0)

predictions_df['predicted_clv'] = final_pred_hybrid


# Load the test_customers template
test_customers_export = pd.read_csv(DATA_DATASETS / "test_customers.csv", sep=";")

test_customers_export = test_customers_export.drop(columns=['churn_23_24', 'revenue_23_24'], errors='ignore')
test_customers_export = test_customers_export.merge(
    predictions_df, on='customer.id', how='left'
)

# Clean up and populate the target column
test_customers_export['churn_23_24'] = test_customers_export['predicted_churn']
test_customers_export['revenue_23_24'] = test_customers_export['predicted_clv']
test_customers_export = test_customers_export.drop(columns=['predicted_churn', 'predicted_clv', 'join_id'], errors='ignore')

# Save the final file
test_customers_export.to_csv(DATA_DATASETS / "test_customers.csv", sep=";", index=False)

Training: 616 aktiv, 6193 inaktiv (10.1:1)


In [11]:
# Take predictions from the single-stage model
pred_single = np.clip(np.array(lgbm_model.predict(X_test_sfs)), 0, None)

# HYBRID: If Stage 1 says "Active", take the baseline prediction. Otherwise, use 0.
final_pred_hybrid = np.where(churn_pred_test_opt == 1, pred_single, 0)

# Calculate MAE for the hybrid model
mae_hybrid = mean_absolute_error(y_test, final_pred_hybrid)
print(f"Hybrid Two-Stage MAE: {mae_hybrid:.2f} NOK")

# Segmented evaluation for the hybrid model
results_hybrid = pd.DataFrame({
    "actual":         y_test.values.ravel(),
    "pred_hybrid":    final_pred_hybrid,
    "error_hybrid":   np.abs(y_test.values.ravel() - final_pred_hybrid)
})
results_hybrid["bucket"] = pd.cut(results_hybrid["actual"],
    bins=[-0.01, 0.01, 1000, 5000, 15000, 30000, 45000, 60000, 999999],
    labels=["Churned (0)", "1–1k NOK", "1k–5k NOK", "5k-15k NOK", "15k-30k NOK", "30k-45k NOK", "45k-60k NOK", "60k+ NOK"])

print("\nMAE for each customer segment (HYBRID):")
print(results_hybrid.groupby("bucket", observed=True).agg(
    n             = ("actual", "count"),
    mae_hybrid    = ("error_hybrid", "mean"),
    avg_actual    = ("actual", "mean"),
).round(2).to_string())

Hybrid Two-Stage MAE: 2610.11 NOK

MAE for each customer segment (HYBRID):
                n  mae_hybrid  avg_actual
bucket                                   
Churned (0)  7245     2610.11         0.0
